# 🤖 ТАРС v4 — Автономное обучение

**Всё через один скрипт `local_train.py`. Всё сохраняется на Google Drive. Переподключение = продолжение.**

| Что | Где |
|---|---|
| Датасеты | `MyDrive/TarsData/` |
| Модели + чекпоинты | `MyDrive/TarsModels/` |
| Логи | `local_train.log` |

### Уровни обучения
| Уровень | Время | Назначение |
|---|---|---|
| `small` | ~15 мин | Быстрый тест, отладка |
| `medium` | ~3 часа | Стандартное обучение |
| `max` | ~15 часов | Продакшн |

### Инструкция
1. **Runtime → Change runtime type → L4 GPU** (или T4 бесплатно)
2. **Выберите уровень** в ячейке ниже и запустите
3. Если отключилось — просто **запустить ячейку заново** (всё продолжится с места остановки)

In [ ]:
#@title 🚀 **ОБУЧЕНИЕ ТАРС** — всё через один скрипт { display-mode: "form" }

#@markdown ### Настройки
LEVEL = "medium" #@param ["small", "medium", "max"] {type:"string"}
#@markdown - `small` — быстрый тест (~15 мин)
#@markdown - `medium` — стандарт (~3 часа)
#@markdown - `max` — продакшн (~15 часов)

SKIP_VOICE = True #@param {type:"boolean"}
SKIP_POSTTRAIN = False #@param {type:"boolean"}

# ============================================================
import os, sys, time, shutil, subprocess
from pathlib import Path

# ==== 1. GOOGLE DRIVE ====
print('═' * 60)
print('  🤖 ТАРС v4 — Автономное обучение')
print('═' * 60)
print()

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA   = Path('/content/drive/MyDrive/TarsData')
DRIVE_MODELS = Path('/content/drive/MyDrive/TarsModels')
DRIVE_DATA.mkdir(parents=True, exist_ok=True)
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)

print(f'  ☁️  Drive: data={DRIVE_DATA}')
print(f'           models={DRIVE_MODELS}')

existing_data = list(DRIVE_DATA.glob('*.txt'))
existing_models = list(DRIVE_MODELS.glob('*.pt'))
if existing_data:
    mb = sum(f.stat().st_size for f in existing_data) / 1024**2
    print(f'  📂 Данные: {len(existing_data)} файлов, {mb:.0f} MB')
if existing_models:
    mb = sum(f.stat().st_size for f in existing_models) / 1024**2
    print(f'  🧠 Модели: {len(existing_models)} файлов, {mb:.0f} MB')
print()

# ==== 2. GPU CHECK ====
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print()

# ==== 3. CLONE / UPDATE CODE ====
os.chdir('/content')
if os.path.exists('TarsSSM-Py'):
    os.chdir('/content/TarsSSM-Py')
    !git pull --rebase 2>/dev/null || true
else:
    !git clone https://github.com/Vazilll/TarsSSM-Py.git
    os.chdir('/content/TarsSSM-Py')

ROOT = Path('/content/TarsSSM-Py')
print(f'  📁 Root: {ROOT}')

# ==== 4. SYMLINKS: data/ → Drive, models/ → Drive ====
def safe_symlink(local, drive_target):
    if local.is_symlink():
        return
    if local.exists():
        for f in local.rglob('*'):
            if f.is_file():
                rel = f.relative_to(local)
                dest = drive_target / rel
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.move(str(f), str(dest))
        shutil.rmtree(str(local))
    local.symlink_to(drive_target)

safe_symlink(ROOT / 'data', DRIVE_DATA)
safe_symlink(ROOT / 'models', DRIVE_MODELS)
print('  🔗 data/ → Drive/TarsData')
print('  🔗 models/ → Drive/TarsModels')
print()

# ==== 5. INSTALL DEPS ====
print('  📦 Установка зависимостей...')
!pip install -q torch einops numpy tqdm sentencepiece datasets psutil 2>/dev/null
print()

# ==== 6. KEEP-ALIVE (анти-таймаут) ====
import threading
def _keep_alive():
    while True:
        time.sleep(600)
        elapsed = time.time() - _train_start
        h, m = int(elapsed // 3600), int((elapsed % 3600) // 60)
        print(f'  ⏰ Keep-alive: {h}h {m}m — обучение продолжается...', flush=True)

_train_start = time.time()
_ka = threading.Thread(target=_keep_alive, daemon=True)
_ka.start()

# ==== 7. CHECKPOINT STATE ====
has_data = len(list(DRIVE_DATA.glob('hf_*.txt'))) > 0 or len(list(DRIVE_DATA.glob('wiki_*.txt'))) > 0

extra_args = []
if has_data:
    extra_args.append('--skip-download')
    print('  ✅ Данные на Drive — пропуск скачивания')

extra_args += ['--resume']  # Всегда --resume для Colab (могут отключить)

if SKIP_VOICE:
    extra_args.append('--skip-voice')
if SKIP_POSTTRAIN:
    extra_args.append('--skip-posttrain')
print()

# ==== 8. ОБУЧЕНИЕ через local_train.py ====
print('═' * 60)
print(f'  🎓 Начало обучения: уровень={LEVEL}')
print('  Всё сохраняется на Drive автоматически')
print('═' * 60)
print(flush=True)

# Маппинг уровней (если пользователь выбрал small)
level_map = {'small': 'small'}
actual_level = level_map.get(LEVEL, LEVEL)

cmd = [
    sys.executable, '-u', str(ROOT / 'local_train.py'),
    '--level', actual_level,
    '--drive', 'colab',
] + extra_args

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

process = subprocess.Popen(
    cmd,
    cwd=str(ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=env,
    bufsize=1,
    universal_newlines=True,
)

for line in process.stdout:
    print(line, end='', flush=True)

returncode = process.wait()

# ==== 9. РЕЗУЛЬТАТ ====
elapsed = time.time() - _train_start
hours = elapsed / 3600
minutes = elapsed / 60

print()
print('═' * 60)
if returncode == 0:
    print(f'  ✅ ОБУЧЕНИЕ ЗАВЕРШЕНО за {hours:.1f}ч ({minutes:.0f} мин)!')
    print()
    for f in sorted(DRIVE_MODELS.rglob('*.pt')):
        mb = f.stat().st_size / 1024**2
        print(f'    💾 {f.name}: {mb:.1f} MB')
    print()
    print('  📁 Данные:  MyDrive/TarsData/')
    print('  🧠 Модели: MyDrive/TarsModels/')
    print('  🚀 Запуск:  python launch_tars.py')
else:
    print(f'  ⚠️  Ошибка (код {returncode})')
    print(f'     Время: {minutes:.0f} мин')
    print()
    print('  Модели сохранены на Drive.')
    print('  ⏯️  Просто перезапустите эту ячейку — обучение продолжится.')
    print('  📋 Логи: local_train.log')
print('═' * 60)

In [ ]:
#@title 📊 Статус обучения { display-mode: "form" }
from pathlib import Path

DRIVE_DATA   = Path('/content/drive/MyDrive/TarsData')
DRIVE_MODELS = Path('/content/drive/MyDrive/TarsModels')

print('═' * 50)
print('  📊 Статус обучения')
print('═' * 50)

if DRIVE_DATA.exists():
    txts = list(DRIVE_DATA.glob('*.txt'))
    total = sum(f.stat().st_size for f in txts) / 1024**2
    print(f'  📁 Данные: {len(txts)} файлов, {total:.0f} MB')

if DRIVE_MODELS.exists():
    for f in sorted(DRIVE_MODELS.rglob('*.pt')):
        mb = f.stat().st_size / 1024**2
        print(f'  🧠 {f.relative_to(DRIVE_MODELS)}: {mb:.1f} MB')

try:
    import torch
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f'  🎮 VRAM: {alloc:.1f}/{total:.1f} GB')
except: pass

print()
print('  📋 Последние строки лога:')
log = Path('/content/TarsSSM-Py/local_train.log')
if log.exists():
    lines = log.read_text(encoding='utf-8', errors='replace').strip().split('\n')
    for l in lines[-15:]:
        print(f'    {l}')
else:
    print('    (нет лога)')
print('═' * 50)

In [ ]:
#@title 📥 Скачать модели локально { display-mode: "form" }
from google.colab import files
from pathlib import Path
import zipfile, os

DRIVE_MODELS = Path('/content/drive/MyDrive/TarsModels')
tars_v3 = DRIVE_MODELS / 'tars_v3'

if tars_v3.exists():
    zip_path = '/content/tars_v3_models.zip'
    print('📦 Упаковка...')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in tars_v3.rglob('*'):
            if f.is_file():
                zf.write(f, f.relative_to(DRIVE_MODELS))
                print(f'  + {f.name} ({f.stat().st_size / 1024**2:.1f} MB)')
    
    total_mb = os.path.getsize(zip_path) / 1024**2
    print(f'\n💾 Итого: {total_mb:.0f} MB')
    print('📥 Скачивание...')
    files.download(zip_path)
else:
    print('⚠️  Модели не найдены. Сначала запустите обучение.')

### 📝 Полезные команды
```python
# Обучить только Mamba-2 (фаза 4)
!cd /content/TarsSSM-Py && python -u local_train.py --level medium --drive colab --phase 4 --resume

# Только CoT (фаза 12)
!cd /content/TarsSSM-Py && python -u local_train.py --level medium --drive colab --phase 12 --resume

# Быстрый тест
!cd /content/TarsSSM-Py && python -u local_train.py --level small --drive colab --resume

# MAX модель (768M params)
!cd /content/TarsSSM-Py && python -u local_train.py --level max --drive colab --resume

# Посмотреть лог
!tail -30 /content/TarsSSM-Py/local_train.log
```